## Text Preprocessing Description 

Each document’s text is processed as follows:

1. Convert all text to lowercase to remove case sensitivity.
2. Remove all punctuation, symbols, and special characters.
3. Preserve all alphabetic characters across languages.
4. Preserve numeric characters when present.
5. Normalize whitespace by collapsing multiple spaces.
6. Tokenize the cleaned text into an ordered list of words, preserving the original word sequence.

The resulting output is stored as an array of words (`preprocessed_words`) and is used as input for  accuracy metrics.


###  Tesseract Preprocessing

In [0]:
df = spark.table("tesseract_accuracy")

In [0]:
from pyspark.sql import functions as F

preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",   # keep letters + numbers
                    ""
                )
            ),
            r"\s+"
        )
    )
)

display(preprocessed_df.select("path", "final_text", "preprocessed_words").limit(50))


In [0]:
display(dbutils.fs.ls("dbfs:/FileStore/accuracy_experiment/"))

In [0]:
(preprocessed_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("tesseract_accuracy"))

In [0]:
display(
    spark.table("tesseract_accuracy")
         .select("path", "final_text", "preprocessed_words")
         .orderBy("path")
)

###  EasyOCR Preprocessing

In [0]:
df = spark.table("easyocr_exp")

In [0]:
from pyspark.sql import functions as F

preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",  
                    ""
                )
            ),
            r"\s+"
        )
    )
)

display(preprocessed_df.select("path", "final_text", "preprocessed_words").limit(50))


In [0]:
(preprocessed_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("easyocr_exp"))

In [0]:
display(
    spark.table("easyocr_exp")
         .select("path", "final_text", "preprocessed_words")
         .orderBy("path")
)

### Azure Preprocessing

In [0]:
df = spark.table("azure_ocr_exp")

In [0]:
from pyspark.sql import functions as F

preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",  
                    ""
                )
            ),
            r"\s+"
        )
    )
)

display(preprocessed_df.select("path", "final_text", "preprocessed_words").limit(50))


In [0]:
(preprocessed_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("azure_ocr_exp"))

In [0]:
display(
    spark.table("azure_ocr_exp")
         .select("path", "final_text", "preprocessed_words")
         .orderBy("path")
)

In [0]:
%sql
DESCRIBE DETAIL azure_ocr_exp;

### GPT-5 Preprocessing

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# =========================
# CONFIG
# =========================
INPUT_PATH = "dbfs:/FileStore/tables/"
OUTPUT_TABLE = "gpt5_exp"   

PREVIEW_CHARS = 500    
PREVIEW_TOKENS = 50    

# =========================
# READ TXT FILES
# =========================
raw_df = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(INPUT_PATH)
)

df = raw_df
if "_metadata" in df.columns:
    df = df.withColumn("path", F.col("_metadata.file_path"))
else:
    df = df.withColumn("path", F.input_file_name())

df = df.withColumn("final_text", F.col("value").cast(StringType())).drop("value")

# =========================
# PREPROCESS
# =========================
preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",   # keep letters + numbers
                    ""
                )
            ),
            r"\s+"
        )
    )
)

# =========================
# SAFE PREVIEW
# =========================
preview_df = (
    preprocessed_df
    .select(
        "path",
        F.length("final_text").alias("final_text_len"),
        F.substring("final_text", 1, PREVIEW_CHARS).alias("final_text_preview"),
        F.size("preprocessed_words").alias("token_count"),
        F.slice("preprocessed_words", 1, PREVIEW_TOKENS).alias("tokens_preview")
    )
    .orderBy("path")
)

display(preview_df.limit(50))

# =========================
# WRITING RESULTS TO DELTA
# =========================
(
    preprocessed_df
    .select("path", "final_text", "preprocessed_words")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(OUTPUT_TABLE)
)

print(f"[SUCCESS] Wrote Delta table: {OUTPUT_TABLE}")

display(
    spark.table(OUTPUT_TABLE)
         .select(
             "path",
             F.length("final_text").alias("final_text_len"),
             F.size("preprocessed_words").alias("token_count")
         )
         .orderBy("path")
)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

### GPT-4 Preprocessing

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# =========================
# CONFIG
# =========================
INPUT_PATH = "dbfs:/FileStore/tables/GPT4"
OUTPUT_TABLE = "gpt4_exp"   

PREVIEW_CHARS = 500    
PREVIEW_TOKENS = 50    

# =========================
# READ TXT FILES
# =========================
raw_df = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(INPUT_PATH)
)

df = raw_df
if "_metadata" in df.columns:
    df = df.withColumn("path", F.col("_metadata.file_path"))
else:
    df = df.withColumn("path", F.input_file_name())

df = df.withColumn("final_text", F.col("value").cast(StringType())).drop("value")

# =========================
# PREPROCESS
# =========================
preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",   # keep letters + numbers
                    ""
                )
            ),
            r"\s+"
        )
    )
)

# =========================
# SAFE PREVIEW
# =========================
preview_df = (
    preprocessed_df
    .select(
        "path",
        F.length("final_text").alias("final_text_len"),
        F.substring("final_text", 1, PREVIEW_CHARS).alias("final_text_preview"),
        F.size("preprocessed_words").alias("token_count"),
        F.slice("preprocessed_words", 1, PREVIEW_TOKENS).alias("tokens_preview")
    )
    .orderBy("path")
)

display(preview_df.limit(50))

# =========================
# WRITING RESULTS TO DELTA
# =========================
(
    preprocessed_df
    .select("path", "final_text", "preprocessed_words")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(OUTPUT_TABLE)
)

print(f"[SUCCESS] Wrote Delta table: {OUTPUT_TABLE}")

display(
    spark.table(OUTPUT_TABLE)
         .select(
             "path",
             F.length("final_text").alias("final_text_len"),
             F.size("preprocessed_words").alias("token_count")
         )
         .orderBy("path")
)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

### Ground Truth Preprocessing

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# =========================
# CONFIG
# =========================
INPUT_PATH = "dbfs:/FileStore/tables/Ground_Truth"
OUTPUT_TABLE = "ground_truth_exp"   

PREVIEW_CHARS = 500    
PREVIEW_TOKENS = 50    

# =========================
# READ TXT FILES
# =========================
raw_df = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(INPUT_PATH)
)

df = raw_df
if "_metadata" in df.columns:
    df = df.withColumn("path", F.col("_metadata.file_path"))
else:
    df = df.withColumn("path", F.input_file_name())

df = df.withColumn("final_text", F.col("value").cast(StringType())).drop("value")

# =========================
# PREPROCESS
# =========================
preprocessed_df = (
    df.withColumn(
        "preprocessed_words",
        F.split(
            F.trim(
                F.regexp_replace(
                    F.lower(F.coalesce(F.col("final_text"), F.lit(""))),
                    r"[^\p{L}\p{N}\s]",   # keep letters + numbers
                    ""
                )
            ),
            r"\s+"
        )
    )
)

# =========================
# SAFE PREVIEW
# =========================
preview_df = (
    preprocessed_df
    .select(
        "path",
        F.length("final_text").alias("final_text_len"),
        F.substring("final_text", 1, PREVIEW_CHARS).alias("final_text_preview"),
        F.size("preprocessed_words").alias("token_count"),
        F.slice("preprocessed_words", 1, PREVIEW_TOKENS).alias("tokens_preview")
    )
    .orderBy("path")
)

display(preview_df.limit(50))

# =========================
# WRITING RESULTS TO DELTA
# =========================
(
    preprocessed_df
    .select("path", "final_text", "preprocessed_words")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(OUTPUT_TABLE)
)

print(f"[SUCCESS] Wrote Delta table: {OUTPUT_TABLE}")

display(
    spark.table(OUTPUT_TABLE)
         .select(
             "path",
             F.length("final_text").alias("final_text_len"),
             F.size("preprocessed_words").alias("token_count")
         )
         .orderBy("path")
)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

### Testing Accuracy

In [0]:
print("OCR rows:", spark.table("azure_ocr_exp").count())
print("GT rows :", spark.table("ground_truth_exp").count())


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

In [0]:
spark.table("azure_ocr_exp").printSchema()
spark.table("ground_truth_exp").printSchema()

spark.table("azure_ocr_exp").selectExpr("typeof(preprocessed_words) as t").show(5, False)
spark.table("ground_truth_exp").selectExpr("typeof(preprocessed_words) as t").show(5, False)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

In [0]:
from pyspark.sql import functions as F

ocr = spark.table("azure_ocr_exp").select("path")
gt  = spark.table("ground_truth_exp").select("path")

print("Distinct OCR paths:", ocr.select("path").distinct().count())
print("Distinct GT  paths:", gt.select("path").distinct().count())

# Show a few examples side-by-side
display(ocr.limit(10))
display(gt.limit(10))


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def basename(c):
    return F.lower(F.trim(F.regexp_extract(F.coalesce(col(c), F.lit("")), r"([^/]+)$", 1)))

def doc_id_from_digits(path_col, min_len=6):
    # Extract a digit sequence with length >= min_len (e.g., 0000005183)
    base = basename(path_col)
    return F.regexp_extract(base, rf"(\d{{{min_len},}})", 1)

ocr_raw = spark.table("azure_ocr_exp").select("path", "final_text", "preprocessed_words") \
    .withColumnRenamed("path", "ocr_path") \
    .withColumnRenamed("final_text", "ocr_text") \
    .withColumnRenamed("preprocessed_words", "ocr_tokens") \
    .withColumn("doc_id", doc_id_from_digits("ocr_path", min_len=6))

gt_raw = spark.table("ground_truth_exp").select("path", "final_text", "preprocessed_words") \
    .withColumnRenamed("path", "gt_path") \
    .withColumnRenamed("final_text", "gt_text") \
    .withColumnRenamed("preprocessed_words", "gt_tokens") \
    .withColumn("doc_id", doc_id_from_digits("gt_path", min_len=6))

# Debug match rate
print("OCR rows:", ocr_raw.count())
print("GT rows :", gt_raw.count())
print("OCR missing doc_id:", ocr_raw.filter(F.length("doc_id")==0).count())
print("GT missing doc_id :", gt_raw.filter(F.length("doc_id")==0).count())

matches = ocr_raw.select("doc_id").where(F.length("doc_id")>0) \
    .join(gt_raw.select("doc_id").where(F.length("doc_id")>0), "doc_id", "inner").count()
print("doc_id matches:", matches)

df = ocr_raw.join(gt_raw, on="doc_id", how="inner")
print("Joined rows:", df.count())
display(df.select("doc_id","ocr_path","gt_path").limit(50))


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType
from collections import Counter
from pyspark.sql.functions import udf

# =========================
# CONFIG
# =========================
OCR_TABLE = "azure_ocr_exp"
GT_TABLE  = "ground_truth_exp"

TEXT_COL  = "final_text"
TOKEN_COL = "preprocessed_words"

MIN_DIGITS_LEN = 6

TOKEN_WINDOW_SIZE = 50
TOKEN_WINDOW_STEP = 25

OUT_PER_DOC_TABLE = "accuracy_results_per_doc"
OUT_SUMMARY_TABLE = "accuracy_results_summary"

# =========================
# HELPERS
# =========================
def basename_expr(path_col):
    return F.lower(F.trim(F.regexp_extract(F.coalesce(path_col, F.lit("")), r"([^/]+)$", 1)))

def doc_id_from_digits_expr(path_col, min_len=6):
    base = basename_expr(path_col)
    return F.regexp_extract(base, rf"(\d{{{min_len},}})", 1)

def normalize_for_cer_expr(text_col):
    return F.regexp_replace(F.lower(F.coalesce(text_col, F.lit(""))), r"[^\p{L}\p{N}]+", "")

# =========================
# 1) LOAD + DOC_ID + JOIN
# =========================
ocr_df = (
    spark.table(OCR_TABLE)
         .select(
             col("path").alias("ocr_path"),
             col(TEXT_COL).alias("ocr_text"),
             col(TOKEN_COL).alias("ocr_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("ocr_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

gt_df = (
    spark.table(GT_TABLE)
         .select(
             col("path").alias("gt_path"),
             col(TEXT_COL).alias("gt_text"),
             col(TOKEN_COL).alias("gt_tokens")
         )
         .withColumn("doc_id", doc_id_from_digits_expr(col("gt_path"), MIN_DIGITS_LEN))
         .filter(F.length("doc_id") > 0)
)

df = ocr_df.join(gt_df, on="doc_id", how="inner")

print("OCR rows:", ocr_df.count())
print("GT rows :", gt_df.count())
print("Joined rows:", df.count())
display(df.select("doc_id", "ocr_path", "gt_path").limit(50))

# =========================
# 2) SAFE WORD BAG-OF-WORDS F1 
# =========================

ocr_counts = (
    df.select("doc_id", F.explode_outer("ocr_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("ocr_cnt"))
)

gt_counts = (
    df.select("doc_id", F.explode_outer("gt_tokens").alias("token"))
      .groupBy("doc_id", "token")
      .agg(F.count("*").alias("gt_cnt"))
)

overlap = (
    ocr_counts.join(gt_counts, on=["doc_id", "token"], how="inner")
              .withColumn("matched_cnt", F.least(col("ocr_cnt"), col("gt_cnt")).cast("double"))
              .groupBy("doc_id")
              .agg(F.sum("matched_cnt").alias("matched_words"))
)

totals = (
    df.select(
        "doc_id",
        F.size("ocr_tokens").cast("double").alias("ocr_word_count"),
        F.size("gt_tokens").cast("double").alias("gt_word_count")
    )
)

df = (
    df.join(overlap, on="doc_id", how="left")
      .join(totals, on="doc_id", how="left")
      .withColumn("matched_words", F.coalesce(col("matched_words"), F.lit(0.0)))
)

df = df.withColumn(
    "word_precision",
    F.when(col("ocr_word_count") > 0, col("matched_words") / col("ocr_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_recall",
    F.when(col("gt_word_count") > 0, col("matched_words") / col("gt_word_count")).otherwise(F.lit(0.0))
)

df = df.withColumn(
    "word_f1",
    F.when(
        (col("word_precision") + col("word_recall")) > 0,
        (F.lit(2.0) * col("word_precision") * col("word_recall")) / (col("word_precision") + col("word_recall"))
    ).otherwise(F.lit(0.0))
)

# =========================
# 3) CER (
# =========================
df = df.withColumn("ocr_norm", normalize_for_cer_expr(col("ocr_text")))
df = df.withColumn("gt_norm",  normalize_for_cer_expr(col("gt_text")))

df = df.withColumn("edit_distance", F.levenshtein(col("gt_norm"), col("ocr_norm")).cast("double"))
df = df.withColumn("gt_char_count", F.length(col("gt_norm")).cast("double"))

df = df.withColumn(
    "cer",
    F.when(col("gt_char_count") > 0, col("edit_distance") / col("gt_char_count")).otherwise(F.lit(None).cast("double"))
)

df = df.withColumn(
    "char_accuracy",
    F.when(col("cer").isNotNull(), F.greatest(F.lit(0.0), F.lit(1.0) - col("cer"))).otherwise(F.lit(None).cast("double"))
)

# =========================
# 4) SLIDING WINDOW TOKEN F1 
# =========================
def bow_f1_from_lists(a, b):
    if a is None or b is None or len(a) == 0 or len(b) == 0:
        return 0.0
    ca = Counter(a)
    cb = Counter(b)
    matched = 0
    for w in ca:
        if w in cb:
            matched += ca[w] if ca[w] < cb[w] else cb[w]
    prec = matched / float(len(a))
    rec  = matched / float(len(b))
    if (prec + rec) == 0.0:
        return 0.0
    return (2.0 * prec * rec) / (prec + rec)

def sliding_window_score(ocr_tokens, gt_tokens, window_size, step_size):
    if ocr_tokens is None or gt_tokens is None or len(ocr_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    ocr_windows = []
    i = 0
    while i < len(ocr_tokens):
        w = ocr_tokens[i:i+window_size]
        if len(w) > 0:
            ocr_windows.append(w)
        i += step_size

    if len(ocr_windows) == 0:
        return 0.0

    scores = []
    j = 0
    while j < len(gt_tokens):
        gt_w = gt_tokens[j:j+window_size]
        if len(gt_w) == 0:
            j += step_size
            continue

        best = 0.0
        for ow in ocr_windows:
            s = bow_f1_from_lists(ow, gt_w)
            if s > best:
                best = s

        scores.append(best)
        j += step_size

    if len(scores) == 0:
        return 0.0
    return float(sum(scores)) / float(len(scores))

@udf(returnType=DoubleType())
def sliding_window_udf(ocr_tokens, gt_tokens):
    return sliding_window_score(ocr_tokens, gt_tokens, TOKEN_WINDOW_SIZE, TOKEN_WINDOW_STEP)

df = df.withColumn("sliding_window_f1", sliding_window_udf(col("ocr_tokens"), col("gt_tokens")))

# =========================
# 5) PER-DOC RESULTS
# =========================
per_doc = df.select(
    "doc_id",
    "ocr_path",
    "gt_path",
    "ocr_word_count", "gt_word_count", "matched_words",
    "word_precision", "word_recall", "word_f1",
    "edit_distance", "gt_char_count", "cer", "char_accuracy",
    "sliding_window_f1"
).withColumn("run_ts", F.current_timestamp())

display(per_doc.orderBy(F.desc("sliding_window_f1")))

# =========================
# 6) SUMMARY RESULTS
# =========================
summary = per_doc.agg(
    F.count("*").alias("doc_count"),

    F.avg("word_f1").alias("avg_word_f1"),
    F.expr("percentile_approx(word_f1, 0.5)").alias("median_word_f1"),

    F.avg("sliding_window_f1").alias("avg_sliding_window_f1"),
    F.expr("percentile_approx(sliding_window_f1, 0.5)").alias("median_sliding_window_f1"),

    F.avg("cer").alias("avg_cer"),
    F.avg("char_accuracy").alias("avg_char_accuracy")
).withColumn("run_ts", F.current_timestamp())

display(summary)

# =========================
# 7) SAVE TO DELTA 
# =========================
(per_doc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_PER_DOC_TABLE))

(summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_SUMMARY_TABLE))

print(f"{OUT_PER_DOC_TABLE}")
print(f"{OUT_SUMMARY_TABLE}")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data

In [0]:
display(
    per_doc.select(
        "doc_id",
        "word_f1",
        "sliding_window_f1",
        "cer",
        "ocr_path",
        "gt_path"
    ).orderBy(F.desc("sliding_window_f1"))
)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:486)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:768)
	at com.data